In [1]:
import pandas as pd
import numpy as np

In [15]:
import pandas as pd

df = pd.read_csv("/content/raw_orders.xlsx - raw_orders.csv")

print("Raw shape:", df.shape)
df.head()

Raw shape: (932, 21)


,order_id,order_date,ship_date,customer_id,customer_name,segment,region,state,city,category,...,product_name,ship_mode,quantity,unit_price,discount,sales,cost,profit,payment_status,order_status
0,ORD-2024-10001,21 Jul 2024,07/27/2024,CUST-4655,Karan Malhotra,Consumer,North,Rajasthan,Jaipur,Office Supplies,...,Pro Paper 122,Second Class,8,2976.45,0.2,19049.28,15841.89,3207.39,Paid,Completed
1,ORD-2024-10002,08/31/2024,05 Sep 2024,CUST-2054,Mira Das,Small Business,East,Odisha,Bhubaneswar,Furniture,...,Pro Tables 258,Same Day,3,2437.50,0,7312.50,4904.73,2407.77,Paid,Completed
2,ORD-2024-10003,06/08/2024,15 Jun 2024,CUST-6843,Isha Nair,Consumer,East,Odisha,Bhubaneswar,Office Supplies,...,Elite Art 525,First Class,4,1065.43,0.55,4261.72,2465.69,1796.03,Paid,Completed
3,ORD-2024-10004,28-11-2024,11/29/2024,CUST-3722,Arjun Bose,Small Business,South,Telangana,Hyderabad,Furniture,...,Smart Furnishings 294,Standard Class,6,1848.43,0.25,8317.93,6796.77,1521.16,Paid,Completed
4,ORD-2025-10005,29 Aug 2025,2025-09-01,CUST-8130,Ananya Rao,Small Business,West,Maharashtra,Mumbai,Furniture,...,Elite Chairs 543,First Class,7,2973.76,0,20816.32,17705.37,3110.95,Paid,Cancelled


In [16]:
print("Missing values per column:\n", df.isnull().sum())
print("\nFully duplicate rows:", df.duplicated().sum())
print("Duplicate order_ids:", df['order_id'].duplicated().sum())
print("\nSample messy categorical values:")
for col in ['segment', 'region', 'payment_status', 'order_status', 'ship_mode']:
    print(col, "->", df[col].dropna().unique()[:8])

Missing values per column:
 order_id           0
order_date         0
ship_date          0
customer_id        0
customer_name      0
segment            0
region            26
state              0
city               0
category           0
sub_category       0
product_name       0
ship_mode         22
quantity           0
unit_price         0
discount          18
sales              0
cost               0
profit             0
payment_status     0
order_status       0
dtype: int64

Fully duplicate rows: 20
Duplicate order_ids: 32

Sample messy categorical values:
segment -> ['Consumer' 'Small Business' 'Corporate' 'Home Office' 'Corporate '
 '  Small Business ' '  Home Office ' '  Consumer ']
region -> ['  North ' 'East' 'South' 'West' 'North' 'NORTH' 'west' 'WEST']
payment_status -> ['Paid' 'Refunded' 'Pending' 'Failed' 'Paid ' 'PENDING' '  Pending '
 'failed']
order_status -> ['Completed' 'Cancelled' 'Returned' 'Completed ' 'completed'
 '  Completed ' '  Cancelled ' 'COMPLETED']
ship_mod

In [7]:
text_cols = ['segment', 'region', 'state', 'city', 'category',
             'sub_category', 'ship_mode', 'payment_status', 'order_status']

for col in text_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()                      # remove leading/trailing spaces
        .str.replace(r'\s+', ' ', regex=True)  # collapse double spaces
        .str.title()                      # standardize casing
        .replace('Nan', np.nan)           # restore true NaNs lost to astype(str)
    )

In [8]:
# The column mixes 4 formats. We detect the pattern per value and
# parse it with the correct format instead of guessing.
def parse_mixed_date(value):
    value = str(value).strip()
    formats = [
        ("%Y-%m-%d", r'^\d{4}-\d{2}-\d{2}$'),        # 2025-09-01
        ("%m/%d/%Y", r'^\d{2}/\d{2}/\d{4}$'),         # 08/31/2024 (US style)
        ("%d-%m-%Y", r'^\d{2}-\d{2}-\d{4}$'),         # 28-11-2024
        ("%d %b %Y", r'^\d{2} [A-Za-z]{3} \d{4}$'),   # 21 Jul 2024
    ]
    import re
    for fmt, pattern in formats:
        if re.match(pattern, value):
            return pd.to_datetime(value, format=fmt, errors='coerce')
    return pd.to_datetime(value, errors='coerce')  # fallback

df['order_date'] = df['order_date'].apply(parse_mixed_date)
df['ship_date'] = df['ship_date'].apply(parse_mixed_date)

# Sanity check: ship_date should never be before order_date
bad_dates = df[df['ship_date'] < df['order_date']]
print("Rows where ship_date < order_date:", len(bad_dates))

Rows where ship_date < order_date: 22


In [9]:
state_to_region = (
    df.dropna(subset=['region'])
      .groupby('state')['region']
      .agg(lambda x: x.mode().iloc[0])
)
df['region'] = df.apply(
    lambda r: state_to_region.get(r['state'], r['region']) if pd.isna(r['region']) else r['region'],
    axis=1
)

In [10]:
# Assumption: unknown ship mode -> label explicitly rather than guess
df['ship_mode'] = df['ship_mode'].fillna('Unknown')

In [11]:
df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
df['discount'] = df['discount'].fillna(0)          # missing -> assume no discount
df['discount'] = df['discount'].clip(lower=0, upper=0.25)  # business rule:
# discounts can't be negative, and policy caps discounts at 25%.
# Values like -0.24 or 0.65 are treated as data entry errors.


In [12]:
before = len(df)
df = df.drop_duplicates()                                   # exact duplicate rows
df = df.drop_duplicates(subset='order_id', keep='first')    # conflicting duplicate IDs
print(f"Removed {before - len(df)} duplicate rows")


Removed 32 duplicate rows


In [13]:
df['calculated_sales'] = (df['quantity'] * df['unit_price'] * (1 - df['discount'])).round(2)
df['sales_mismatch'] = (df['calculated_sales'] - df['sales']).abs() > 1
print("Rows where reported sales disagrees with formula:", df['sales_mismatch'].sum())

Rows where reported sales disagrees with formula: 44


In [ ]:
print("Final shape:", df.shape)
print(df.isnull().sum())

df.to_csv("data/cleaned_orders.csv", index=False)
print("Saved cleaned_orders.csv")
